In [53]:
import torch
from torch.autograd.functional import jacobian
from torch import tensor

In [54]:

def f1(x):
    def jvp(v):
        return v*torch.exp(x)
    return jvp, torch.exp(x)

def f2(x):
    def jvp(v):
        return v*(1-torch.tanh(x)*torch.tanh(x))
    return jvp, torch.tanh(x)

def f3(x):
    def jvp(v):
        return torch.ones_like(x).dot(v.T)
    return jvp, torch.sum(x)

x = tensor([1., 2, 3, 4])
es = torch.eye(len(x), len(x))
fs = [f1, f2, f3]
o = None

jab = []
for v in es:
    xi = x
    for f in fs:
        jvp, o = f(xi)
        xi = o
        v = jvp(v)
    jab.append(v)
jab = torch.column_stack(jab)

print('output:', o)
print('jacobian:\n', jab)

output: tensor(3.9913)
jacobian:
 tensor([[4.6937e-02, 1.1451e-05, 0.0000e+00, 0.0000e+00]])


In [55]:
def f(x): return f3(f2(f1(x)[1])[1])[1]
_o = f(x)
_jab = jacobian(f, x)

print('output:', o)
print('jacobian:\n', jab)

torch.allclose(o, _o)
torch.allclose(jab, _jab)

output: tensor(3.9913)
jacobian:
 tensor([[4.6937e-02, 1.1451e-05, 0.0000e+00, 0.0000e+00]])


True

In [56]:
def f3(x):
    def vjp(v):
        return v*torch.ones_like(x)
    return vjp, torch.sum(x)

x = tensor([1., 2, 3, 4])
xi = x
os = [x]
fs = [f1, f2, f3]
for f in fs:
    _, o = f(xi)
    os.append(o)
    xi = o
os = os[:-1]

dim = len(o) if o.dim() != 0 else 1
es = torch.eye(dim, dim)
jab = []
for v in es:
    for i in reversed(range(len(fs))):
        vjp, _ = fs[i](os[i])
        v = vjp(v)
    jab.append(v)
jab = torch.row_stack(jab)

print('output:', o)
print('jacobian:\n', jab)


output: tensor(3.9913)
jacobian:
 tensor([[4.6937e-02, 1.1451e-05, 0.0000e+00, 0.0000e+00]])


In [57]:
def f(x): return f3(f2(f1(x)[1])[1])[1]
_o = f(x)
_jab = jacobian(f, x)

print('output:', o)
print('jacobian:\n', jab)

torch.allclose(o, _o)
torch.allclose(jab, _jab)

output: tensor(3.9913)
jacobian:
 tensor([[4.6937e-02, 1.1451e-05, 0.0000e+00, 0.0000e+00]])


True